# Session 2 — A/B Testing & Canary Deployments

**Goal:** Learn how to safely roll out new model versions using traffic splitting.

**Canary deployment**: Send a small % of traffic to the new model before full rollout.
- Risk: If the canary has issues, only 10% of users are affected
- Monitoring window: Watch metrics for 24h before promoting to 100%

**A/B test**: Compare two model versions on business metrics (not just offline F1).
- Split: 50/50 traffic to version A and version B
- Measure: Which version drives better customer retention outcomes?

In [0]:
%run ../utils/config

In [0]:

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

endpoint_name = f"churn_{safe_username}"
model_name    = f"{catalog}.{schema}.churn_classifier"

# Check what versions we have
from mlflow import MlflowClient
mlflow_client = MlflowClient()
mlflow_client.set_registry_uri = lambda x: None

import mlflow
mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()
versions = client.search_model_versions(f"name='{model_name}'")
print(f"Available versions for {model_name}:")
for v in versions:
    # search_model_versions doesn't include aliases; use get_model_version instead
    mv = client.get_model_version(model_name, v.version)
    alias_list = mv.aliases if not callable(mv.aliases) else []
    aliases = ", ".join(alias_list) if alias_list else "(none)"
    print(f"  v{v.version}  aliases=[{aliases}]  status={v.status}")

## Deploy as a Canary (90% v1 / 10% v2)

> **Pre-requisite:** You need at least 2 registered versions.
> If you only have 1, run `04_mlflow_experiment.ipynb` again to create a second version.

In [0]:
# Get the two most recent versions
all_versions = sorted(versions, key=lambda v: int(v.version))
if len(all_versions) < 2:
    raise ValueError("Need at least 2 model versions. Re-run 04_mlflow_experiment.ipynb first.")

v1 = all_versions[-2].version  # Previous version (stable)
v2 = all_versions[-1].version  # New version (canary)

print(f"Stable version  : v{v1}")
print(f"Canary version  : v{v2}")
print(f"\nDeploying: 90% to v{v1}, 10% to v{v2}")

# Note: Traffic config API varies by Databricks runtime version.
# Use the served_models approach with traffic_config
endpoint_config = {
    "served_models": [
        {
            "model_name": model_name,
            "model_version": str(v1),
            "workload_size": "Small",
            "scale_to_zero_enabled": True,
            "name": f"v{v1}",
        },
        {
            "model_name": model_name,
            "model_version": str(v2),
            "workload_size": "Small",
            "scale_to_zero_enabled": True,
            "name": f"v{v2}",
        },
    ],
    "traffic_config": {
        "routes": [
            {"served_model_name": f"v{v1}", "traffic_percentage": 90},
            {"served_model_name": f"v{v2}", "traffic_percentage": 10},
        ]
    }
}
print("\nEndpoint config ready. Review before applying:")
import json
print(json.dumps(endpoint_config, indent=2))

In [0]:
# Apply the canary configuration
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedModelInput, TrafficConfig, Route

w.serving_endpoints.update_config_and_wait(
    name=endpoint_name,
    served_models=[
        ServedModelInput(name=f"v{v1}", model_name=model_name, model_version=str(v1),
                         workload_size="Small", scale_to_zero_enabled=True),
        ServedModelInput(name=f"v{v2}", model_name=model_name, model_version=str(v2),
                         workload_size="Small", scale_to_zero_enabled=True),
    ],
    traffic_config=TrafficConfig(routes=[
        Route(served_model_name=f"v{v1}", traffic_percentage=90),
        Route(served_model_name=f"v{v2}", traffic_percentage=10),
    ])
)
print(f"✓ Canary deployed: 90% v{v1} / 10% v{v2}")

## Exercise: Promote to 50/50

Imagine 24 hours have passed. The canary looks healthy — no elevated error rates.
Update the traffic split to 50/50.

**Modify the cell below and run it:**

In [0]:
# Exercise: Change traffic_percentage values to create a 50/50 split
# Hint: Change 90 → 50 and 10 → 50

w.serving_endpoints.update_config_and_wait(
    name=endpoint_name,
    served_models=[
        ServedModelInput(name=f"v{v1}", model_name=model_name, model_version=str(v1),
                         workload_size="Small", scale_to_zero_enabled=True),
        ServedModelInput(name=f"v{v2}", model_name=model_name, model_version=str(v2),
                         workload_size="Small", scale_to_zero_enabled=True),
    ],
    traffic_config=TrafficConfig(routes=[
        Route(served_model_name=f"v{v1}", traffic_percentage=90),  # ← Change to 50
        Route(served_model_name=f"v{v2}", traffic_percentage=10),  # ← Change to 50
    ])
)
print("Traffic split updated!")

## Rollback: Return to 100% v1

If the canary starts showing problems, rollback is instant:

In [0]:
# Rollback to 100% v1
w.serving_endpoints.update_config_and_wait(
    name=endpoint_name,
    served_models=[
        ServedModelInput(name=f"v{v1}", model_name=model_name, model_version=str(v1),
                         workload_size="Small", scale_to_zero_enabled=True),
    ],
    traffic_config=TrafficConfig(routes=[
        Route(served_model_name=f"v{v1}", traffic_percentage=100),
    ])
)
print(f"✓ Rolled back to 100% v{v1}")